# 1. The original model

My earlier course project trained models to sort tweets into **hate speech**, **offensive language** or **neither**. This notebook is that project, cleaned up so it runs from this repository. The method is the same; the fixes are listed below.

**Content note:** the tweets contain slurs and abusive language. This notebook doesn't print any of them.

**What changed from the original**

- **Data and paths.** The data now comes from `get_data.sh` instead of Kaggle folders. The tweets are the Davidson et al. (2017) dataset.
- **One fixed split.** The same stratified 60/20/20 split (seed 42) as [notebook 2](02_safety_audit.ipynb), so the numbers can be compared. The original used an unseeded 67/33 split, so its scores changed on every run.
- **Speed.** Looking up each word in `list(model.index_to_key)` searched a 1.2-million-word list every time. A dictionary lookup gives the same vectors in seconds.
- **Bugs.** A helper called `model` was overwritten by the GloVe model, so later cells crashed. Cells for a missing file (`test_new.csv`) and duplicate cells were removed.
- **Unsupervised baselines scored fairly.** LDA topics and k-means clusters are numbered arbitrarily, so comparing cluster 0 with class 0 is meaningless. Each cluster is now matched to the label it fits best before scoring.
- **Dependencies.** gensim's LDA is replaced with scikit-learn's, and NLTK's tokenizer with a plain split after punctuation is removed; the results are equivalent for this data. `max_features="auto"` became `"sqrt"`, which is what `"auto"` meant for classifiers.

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import re
import string

sys.path.insert(0, str(Path("..", "src").resolve()))
warnings.filterwarnings("ignore")

from sklearn.cluster import KMeans
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, CountVectorizer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
from scipy.optimize import linear_sum_assignment

from ald import data, glove
from ald.config import CLASSES, SEED

try:
    from xgboost import XGBClassifier
    boosting_name, boosting = "XGBoost", XGBClassifier(random_state=SEED)
except ImportError:  # same idea, from scikit-learn
    from sklearn.ensemble import HistGradientBoostingClassifier
    boosting_name, boosting = "Gradient boosting (XGBoost not installed)", HistGradientBoostingClassifier(random_state=SEED)

## Data

24,783 tweets, each labelled by crowd workers. Most are "offensive", and only about 6% are hate speech.

In [2]:
df = data.load_davidson()
train, val, test = data.split(df)
pd.DataFrame({"tweets": df.label.value_counts().sort_index().values,
              "share": df.label.value_counts(normalize=True).sort_index().round(3).values}, index=CLASSES)

,tweets,share
hate,1430,0.058
offensive,19190,0.774
neither,4163,0.168


## Preprocessing

The original function: remove URLs, mentions and hashtags, punctuation and capitals, then drop stopwords.

In [3]:
STOPWORDS = set(ENGLISH_STOP_WORDS)

def preprocess_tweet(tweet):
    # Remove URLs, mentions and hashtags
    tweet = re.sub(r'http\S+|www\S+|https\S+|@\S+|#\S+|"|\n|\t', "", tweet)
    # Remove punctuation and convert to lowercase
    tweet = tweet.translate(str.maketrans("", "", string.punctuation)).lower()
    # Tokenize and remove stop words
    return [token for token in tweet.split() if token not in STOPWORDS]

train_tokens = [preprocess_tweet(t) for t in train.tweet]
test_tokens = [preprocess_tweet(t) for t in test.tweet]
y_train, y_test = train.label.to_numpy(), test.label.to_numpy()

## Shared evaluation helper

The original project reported **weighted** averages, which weight each class by its size. `evaluate` keeps that.

In [4]:
results = {}

def evaluate(name, y_true, y_pred):
    results[name] = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision (weighted)": precision_score(y_true, y_pred, average="weighted", zero_division=0),
        "recall (weighted)": recall_score(y_true, y_pred, average="weighted", zero_division=0),
        "F1 (weighted)": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }
    return pd.Series(results[name], name=name).map("{:.1%}".format)

def match_clusters(clusters_train, y_train, clusters_test):
    """Give each cluster the label it overlaps with most on the training set (one label per cluster)."""
    overlap = confusion_matrix(y_train, clusters_train, labels=[0, 1, 2])
    labels, cluster_ids = linear_sum_assignment(-overlap)
    mapping = dict(zip(cluster_ids, labels))
    return np.array([mapping[c] for c in clusters_test])

## Unsupervised baseline 1: LDA topics on bag-of-words

Can three topics, found without labels, separate the three classes?

In [5]:
bow = CountVectorizer(analyzer=lambda tokens: tokens)
X_bow_train, X_bow_test = bow.fit_transform(train_tokens), bow.transform(test_tokens)
lda = LatentDirichletAllocation(n_components=3, learning_method="online", batch_size=100,
                                max_iter=10, random_state=10).fit(X_bow_train)
topics_train, topics_test = lda.transform(X_bow_train).argmax(1), lda.transform(X_bow_test).argmax(1)
evaluate("LDA topics", y_test, match_clusters(topics_train, y_train, topics_test))

accuracy                60.3%
precision (weighted)    71.7%
recall (weighted)       60.3%
F1 (weighted)           65.1%
Name: LDA topics, dtype: str

## GloVe tweet embeddings

Each tweet becomes the average of its words' 25-dimensional GloVe Twitter vectors (Pennington et al., 2014). Tweets with no known words become all zeros, as in the original.

In [6]:
vectors = glove.load()

def embed(tokens):
    known = [vectors[w] for w in tokens if w in vectors]
    return np.mean(known, axis=0) if known else np.zeros(25, dtype=np.float32)

X_train = np.array([embed(t) for t in train_tokens])
X_test = np.array([embed(t) for t in test_tokens])
X_train.shape, X_test.shape

((14869, 25), (4957, 25))

## Unsupervised baseline 2: k-means on embeddings

In [7]:
kmeans = KMeans(n_clusters=3, init="k-means++", n_init=10, random_state=SEED).fit(X_train)
evaluate("k-means clusters", y_test, match_clusters(kmeans.labels_, y_train, kmeans.predict(X_test)))

accuracy                49.2%
precision (weighted)    72.6%
recall (weighted)       49.2%
F1 (weighted)           56.1%
Name: k-means clusters, dtype: str

## Supervised models

The four classifiers compared in the original project, with the same settings.

In [8]:
for name, clf in [("SVM (default)", SVC(random_state=SEED)),
                  ("Random forest (depth 2)", RandomForestClassifier(max_depth=2, random_state=0)),
                  (boosting_name, boosting),
                  ("AdaBoost (2 estimators)", AdaBoostClassifier(n_estimators=2, random_state=0))]:
    clf.fit(X_train, y_train)
    evaluate(name, y_test, clf.predict(X_test))

### Tuning the SVM

The original grid search, on the training set only. It takes a few minutes.

In [9]:
search = GridSearchCV(SVC(random_state=SEED), {"kernel": ["linear", "rbf"], "C": [1, 10]}, n_jobs=-1)
search.fit(X_train, y_train)
search.best_params_

{'C': 10, 'kernel': 'rbf'}

In [10]:
svm = search.best_estimator_
y_pred_svm = svm.predict(X_test)
evaluate("SVM (tuned)", y_test, y_pred_svm)

accuracy                85.9%
precision (weighted)    83.8%
recall (weighted)       85.9%
F1 (weighted)           83.2%
Name: SVM (tuned), dtype: str

### Tuned random forest

The original grid search over 60 settings found 500 trees, depth 8, `sqrt` features and Gini impurity. It takes over an hour, so this cell uses those settings directly.

In [11]:
rf = RandomForestClassifier(n_estimators=500, max_depth=8, max_features="sqrt", criterion="gini",
                            random_state=42, n_jobs=-1).fit(X_train, y_train)
evaluate("Random forest (tuned)", y_test, rf.predict(X_test))

accuracy                85.2%
precision (weighted)    84.4%
recall (weighted)       85.2%
F1 (weighted)           82.3%
Name: Random forest (tuned), dtype: str

## Results, as the original project reported them

In [12]:
pd.DataFrame(results).T.sort_values("accuracy", ascending=False).style.format("{:.1%}")

,accuracy,precision (weighted),recall (weighted),F1 (weighted)
SVM (tuned),85.9%,83.8%,85.9%,83.2%
SVM (default),85.7%,86.1%,85.7%,82.8%
Gradient boosting (XGBoost not installed),85.5%,82.7%,85.5%,83.3%
Random forest (tuned),85.2%,84.4%,85.2%,82.3%
AdaBoost (2 estimators),81.9%,76.5%,81.9%,79.0%
Random forest (depth 2),81.4%,76.9%,81.4%,76.0%
LDA topics,60.3%,71.7%,60.3%,65.1%
k-means clusters,49.2%,72.6%,49.2%,56.1%


The tuned SVM looks like a solid classifier: about 86% accuracy (the original's unseeded split gave 86.8%). Topics and clusters, found without labels, do no better than labelling every tweet "offensive", which would score 77%.

## What the weighted scores hid

Splitting the tuned SVM's results by class shows the problem.

In [13]:
recall_by_class = recall_score(y_test, y_pred_svm, labels=[0, 1, 2], average=None)
counts = confusion_matrix(y_test, y_pred_svm, labels=[0, 1, 2])
pd.DataFrame(counts, index=[f"true {c}" for c in CLASSES], columns=[f"predicted {c}" for c in CLASSES]) \
  .assign(recall=[f"{r:.1%}" for r in recall_by_class])

,predicted hate,predicted offensive,predicted neither,recall
true hate,6,244,36,2.1%
true offensive,2,3703,134,96.5%
true neither,3,279,550,66.1%


The model caught almost none of the hate speech. Most hateful tweets were labelled "offensive", and weighted averages hid this because the offensive class is 77% of the data.

[Notebook 2](02_safety_audit.ipynb) starts from here: it builds a better model, then asks who each model fails.